# QPEI Round 4 — From Scratch
## Demographics Exploration → Indicator Screening Summary → School-Level Construction Path → Sensitivity Skeleton

This notebook starts **from scratch** relative to the psychometric Round 3 work.  
It does **not** re-run EFA/CFA. It consumes Round 3 outputs where available and builds the next layer of the publishable pipeline.

### Goals

1. **Demographics & sample structure exploration**  
   Respondent and school characteristics across Teacher / Parent / Student / Observation / Environment instruments.

2. **Data-quality and coverage maps**  
   Who responded where; missingness by school and instrument; balance across schools.

3. **Indicator screening summary** (using Round 3 decision log if present)  
   Clean retain / review / drop overview for the paper.

4. **School-level aggregation skeleton**  
   Respondent → school means for retained items → indicator scores → domain scores → weighted QPEI.

5. **Sensitivity analysis skeleton**  
   Theoretical weights vs equal weights vs alternatives; rank stability.

### Design rules (unchanged)

- Six domains remain **theory-defined** formative components.
- Weights: D1 20% | D2 15% | D3 15% | D4 20% | D5 15% | D6 15%.
- EFA/CFA evidence supports indicator decisions; it does **not** redefine the domains.
- No automatic Factor → Domain assignment.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================

!pip -q install openpyxl statsmodels scikit-learn scipy seaborn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, re, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path("/content/drive/MyDrive/QPEI/QPE_MASTER_Cleaned_IDs_Translated_Analysis.xlsx")
R3_RESULTS = Path("/content/drive/MyDrive/QPEI/results_round3.json")
R3_DIR = Path("/content/drive/MyDrive/QPEI/qpei_validation_round3")
OUTPUT_DIR = Path("/content/drive/MyDrive/QPEI/qpei_round4_demographics_construction")

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"

for p in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("QPEI ROUND 4 — DEMOGRAPHICS + CONSTRUCTION PATH")
print("=" * 100)
print("Data       :", DATA_PATH, "| exists:", DATA_PATH.exists())
print("R3 results :", R3_RESULTS, "| exists:", R3_RESULTS.exists())
print("R3 dir     :", R3_DIR, "| exists:", R3_DIR.exists())
print("Output     :", OUTPUT_DIR)
print("Started    :", datetime.now().isoformat(timespec="seconds"))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Master workbook not found: {DATA_PATH}")


# 1. Load workbook and profile every sheet

We inventory all columns, dtypes, and non-null counts so demographic fields can be discovered rather than hard-coded.


In [ ]:
# ============================================================
# 1. LOAD + SHEET PROFILE
# ============================================================

xl = pd.ExcelFile(DATA_PATH)
print("Sheets:")
for s in xl.sheet_names:
    print(" •", s)

sheets = {s: pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names}

profile_rows = []
for name, df in sheets.items():
    for col in df.columns:
        s = df[col]
        profile_rows.append({
            "sheet": name,
            "column": str(col),
            "dtype": str(s.dtype),
            "n_rows": len(s),
            "n_nonnull": int(s.notna().sum()),
            "n_unique": int(s.nunique(dropna=True)),
            "sample_values": " | ".join(
                str(v)[:40] for v in s.dropna().astype(str).unique()[:5]
            ),
        })

profile = pd.DataFrame(profile_rows)
display(profile.head(40))
profile.to_csv(TABLE_DIR / "R4_Table_01_Sheet_Column_Profile.csv", index=False)
print(f"\nTotal columns across all sheets: {len(profile)}")


# 2. Discover demographic and ID columns

Heuristic keyword search across all sheets for likely demographic, geographic, and identifier fields.  
Nothing is assumed; candidates are listed for review and then used in exploration.


In [ ]:
# ============================================================
# 2. DISCOVER DEMOGRAPHIC / ID / CONTEXT COLUMNS
# ============================================================

DEMO_KEYWORDS = [
    r"school", r"sid", r"sch", r"emis",
    r"district", r"upazila", r"union", r"division", r"geo", r"location", r"area",
    r"urban", r"rural", r"type", r"category", r"ownership", r"govt", r"private",
    r"gender", r"sex", r"age", r"grade", r"class", r"section",
    r"teacher", r"parent", r"student", r"respondent",
    r"education", r"edu", r"qualification", r"experience", r"years",
    r"income", r"occupation", r"job", r"socio", r"ses",
    r"enrollment", r"enrol", r"pupil", r"size", r"shift",
    r"enumerator", r"date", r"wave", r"round",
    r"id$", r"_id", r"code",
]

pattern = re.compile("|".join(DEMO_KEYWORDS), re.I)

demo_candidates = profile[
    profile["column"].apply(lambda c: bool(pattern.search(str(c))))
].copy()

# Exclude pure psychometric item columns (tq1, pq2, …)
demo_candidates = demo_candidates[
    ~demo_candidates["column"].str.fullmatch(r"(tq|pq|sq|co|se)\d+", case=False)
]

print("Candidate demographic / ID / context columns:")
display(demo_candidates.sort_values(["sheet", "column"]))
demo_candidates.to_csv(TABLE_DIR / "R4_Table_02_Demographic_Column_Candidates.csv", index=False)


In [ ]:
# ============================================================
# 2A. INSTRUMENT ITEM SETS + ANALYSIS COPIES
# ============================================================

INSTRUMENTS = {
    "Teacher":     {"sheet": "Teacher_Survey",        "prefix": "tq"},
    "Parent":      {"sheet": "Parent_Survey",         "prefix": "pq"},
    "Student":     {"sheet": "Student_Questionnaire", "prefix": "sq"},
    "Observation": {"sheet": "Classroom_Observation", "prefix": "co"},
    "Environment": {"sheet": "School_Environment",    "prefix": "se"},
}

ITEMS = {}
for label, spec in INSTRUMENTS.items():
    df = sheets[spec["sheet"]]
    cols = [
        c for c in df.columns
        if re.fullmatch(fr"{spec['prefix']}\d+", str(c).strip().lower())
    ]
    cols = sorted(cols, key=lambda x: int(re.search(r"\d+", str(x)).group()))
    ITEMS[label] = cols
    print(f"{label:12s}: {len(cols):2d} items")

# Clean analysis copies: 88/99 → NaN for item columns only
analysis = {}
for name, df in sheets.items():
    x = df.copy()
    for col in x.columns:
        if re.fullmatch(r"(tq|pq|sq|co|se)\d+", str(col).strip().lower()):
            x[col] = pd.to_numeric(x[col], errors="coerce")
            x.loc[x[col].isin([88, 99]), col] = np.nan
    analysis[name] = x


# 3. School ID resolution and sample structure

Locate the school identifier on each instrument sheet and summarise respondents per school.


In [ ]:
# ============================================================
# 3. SCHOOL ID RESOLUTION
# ============================================================

def find_school_id_col(df):
    """Return best school-id column name or None."""
    candidates = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if re.search(r"school[ _-]?id|sch[ _-]?id|^sid$|^school$", cl):
            candidates.append((0, c))  # highest priority
        elif re.search(r"school", cl) and df[c].nunique(dropna=True) < 80:
            candidates.append((1, c))
    if not candidates:
        return None
    candidates.sort()
    return candidates[0][1]


school_id_map = {}
school_counts = {}

for label, spec in INSTRUMENTS.items():
    df = analysis[spec["sheet"]]
    sid = find_school_id_col(df)
    school_id_map[label] = sid
    if sid is None:
        print(f"{label:12s}: NO school-id column found")
        school_counts[label] = pd.DataFrame(columns=["school_id", "n_respondents"])
        continue
    counts = (
        df.groupby(sid, dropna=False)
        .size()
        .reset_index(name="n_respondents")
        .rename(columns={sid: "school_id"})
        .sort_values("n_respondents", ascending=False)
    )
    school_counts[label] = counts
    print(f"{label:12s}: school-id = '{sid}'  |  schools = {len(counts)}  |  "
          f"median n = {counts['n_respondents'].median():.0f}  |  "
          f"total N = {counts['n_respondents'].sum()}")
    counts.to_csv(TABLE_DIR / f"R4_School_Counts_{label}.csv", index=False)

print("\nSchool-id map:", school_id_map)


In [ ]:
# ============================================================
# 3A. CROSS-INSTRUMENT SCHOOL COVERAGE MATRIX
# ============================================================

all_schools = set()
for label, counts in school_counts.items():
    if len(counts):
        all_schools |= set(counts["school_id"].astype(str))

coverage = pd.DataFrame({"school_id": sorted(all_schools)})
for label, counts in school_counts.items():
    if len(counts) == 0:
        coverage[label] = 0
        continue
    m = counts.set_index(counts["school_id"].astype(str))["n_respondents"]
    coverage[label] = coverage["school_id"].map(m).fillna(0).astype(int)

coverage["n_instruments_present"] = (coverage[[c for c in coverage.columns if c != "school_id"]] > 0).sum(axis=1)
coverage = coverage.sort_values(["n_instruments_present", "school_id"], ascending=[False, True])

display(coverage)
coverage.to_csv(TABLE_DIR / "R4_Table_03_School_Instrument_Coverage.csv", index=False)

print("\nSchools with all 5 instruments:",
      int((coverage["n_instruments_present"] == 5).sum()))
print("Schools with ≥3 instruments:",
      int((coverage["n_instruments_present"] >= 3).sum()))


# 4. Demographics exploration

For each instrument, summarise every non-item column that looks demographic or contextual.  
Figures are saved for the paper’s sample-description section.


In [ ]:
# ============================================================
# 4. DEMOGRAPHICS EXPLORATION PER INSTRUMENT
# ============================================================

def is_item_col(c):
    return bool(re.fullmatch(r"(tq|pq|sq|co|se)\d+", str(c).strip().lower()))


def summarise_column(s, name):
    """Return a small summary dict for a Series."""
    out = {
        "column": name,
        "n": int(s.notna().sum()),
        "missing_pct": round(100 * s.isna().mean(), 1),
        "n_unique": int(s.nunique(dropna=True)),
    }
    if pd.api.types.is_numeric_dtype(s):
        out["mean"] = round(float(s.mean()), 2) if s.notna().any() else np.nan
        out["sd"] = round(float(s.std()), 2) if s.notna().sum() > 1 else np.nan
        out["min"] = float(s.min()) if s.notna().any() else np.nan
        out["max"] = float(s.max()) if s.notna().any() else np.nan
        out["type"] = "numeric"
    else:
        top = s.astype(str).value_counts(dropna=True).head(8)
        out["top_categories"] = "; ".join(f"{k} ({v})" for k, v in top.items())
        out["type"] = "categorical"
    return out


demo_summaries = []

for label, spec in INSTRUMENTS.items():
    df = analysis[spec["sheet"]]
    non_item = [c for c in df.columns if not is_item_col(c)]
    print("\n" + "=" * 90)
    print(f"{label} — non-item columns ({len(non_item)})")

    rows = []
    for c in non_item:
        row = summarise_column(df[c], str(c))
        row["instrument"] = label
        rows.append(row)
        demo_summaries.append(row)

    summary_df = pd.DataFrame(rows)
    display(summary_df)
    summary_df.to_csv(TABLE_DIR / f"R4_Demo_Summary_{label}.csv", index=False)

    # Plot top categorical columns (≤15 unique, ≥10 non-null)
    cat_cols = [
        c for c in non_item
        if df[c].nunique(dropna=True) <= 15
        and df[c].notna().sum() >= 10
        and not pd.api.types.is_numeric_dtype(df[c])
    ][:6]  # limit figures

    for c in cat_cols:
        vc = df[c].astype(str).value_counts(dropna=True).head(12)
        fig, ax = plt.subplots(figsize=(7, 3.5))
        vc.plot(kind="barh", ax=ax, color="#4C72B0")
        ax.set_title(f"{label}: {c}")
        ax.set_xlabel("Count")
        fig.tight_layout()
        safe = re.sub(r"[^A-Za-z0-9_]+", "_", str(c))[:40]
        fig.savefig(FIGURE_DIR / f"R4_Demo_{label}_{safe}.png", dpi=130)
        plt.show()

    # Numeric demographic-like columns
    num_cols = [
        c for c in non_item
        if pd.api.types.is_numeric_dtype(df[c])
        and df[c].nunique(dropna=True) > 3
        and df[c].notna().sum() >= 10
        and not is_item_col(c)
    ][:4]

    for c in num_cols:
        fig, ax = plt.subplots(figsize=(6, 3.5))
        df[c].dropna().hist(ax=ax, bins=15, color="#55A868", edgecolor="white")
        ax.set_title(f"{label}: {c}")
        fig.tight_layout()
        safe = re.sub(r"[^A-Za-z0-9_]+", "_", str(c))[:40]
        fig.savefig(FIGURE_DIR / f"R4_DemoNum_{label}_{safe}.png", dpi=130)
        plt.show()

demo_all = pd.DataFrame(demo_summaries)
demo_all.to_csv(TABLE_DIR / "R4_Table_04_All_Demographic_Summaries.csv", index=False)


# 5. School master / context table

If a `School_Master` (or similar) sheet exists, profile it fully — this is the natural place for school-level demographics (location, type, enrollment, etc.).


In [ ]:
# ============================================================
# 5. SCHOOL MASTER / CONTEXT
# ============================================================

school_master_name = None
for cand in ["School_Master", "SchoolMaster", "Schools", "School_Info", "school_master"]:
    if cand in sheets:
        school_master_name = cand
        break

if school_master_name is None:
    # fallback: any sheet with 'school' and few rows
    for name, df in sheets.items():
        if re.search(r"school", name, re.I) and len(df) <= 100 and len(df) >= 5:
            school_master_name = name
            break

if school_master_name:
    sm = analysis[school_master_name]
    print(f"School master sheet: {school_master_name}  |  shape = {sm.shape}")
    display(sm.head(15))
    sm.to_csv(TABLE_DIR / "R4_Table_05_School_Master.csv", index=False)

    # Quick value counts for low-cardinality columns
    for c in sm.columns:
        nuniq = sm[c].nunique(dropna=True)
        if 1 < nuniq <= 12:
            print(f"\n--- {c} ---")
            print(sm[c].value_counts(dropna=False).to_string())
else:
    print("No obvious School_Master sheet found. Coverage matrix above is the main school structure view.")
    sm = None


# 6. Consume Round 3 decision log (if available)

If Round 3 has been run, load the item decision log and crosswalk so construction uses only approved indicators.  
If not yet available, the notebook continues with a placeholder retain-all path and flags that substantive review is still required.


In [ ]:
# ============================================================
# 6. LOAD ROUND 3 DECISION LOG + CROSSWALK
# ============================================================

decision_log = None
crosswalk = None

decision_candidates = [
    R3_DIR / "decisions" / "R3_Item_Decision_Log.csv",
    R3_DIR / "tables" / "R3_Table_17_Item_Decision_Flags.csv",
    Path("/content/drive/MyDrive/QPEI/qpei_validation_round2/tables/R2_Table_17_Item_Decision_Flags.csv"),
]

for p in decision_candidates:
    if p.exists():
        decision_log = pd.read_csv(p)
        print(f"Loaded decision log: {p}  |  rows = {len(decision_log)}")
        break

crosswalk_candidates = [
    R3_DIR / "tables" / "R3_Table_13_Empirical_QPEI_Crosswalk_REVIEW.csv",
    Path("/content/drive/MyDrive/QPEI/qpei_validation_round2/tables/R2_Table_13_Empirical_QPEI_Crosswalk_REVIEW.csv"),
]

for p in crosswalk_candidates:
    if p.exists():
        crosswalk = pd.read_csv(p)
        print(f"Loaded crosswalk: {p}  |  rows = {len(crosswalk)}")
        break

if decision_log is not None:
    display(decision_log.head(20))
    if "decision" in decision_log.columns:
        print("\nDecision counts:")
        print(decision_log.groupby(["instrument", "decision"]).size().unstack(fill_value=0))
else:
    print("No Round 3/2 decision log found. Construction will use all items as provisional RETAIN pending review.")


# 7. QPEI framework and provisional item → domain map

Domains and weights are fixed.  
Until the substantive crosswalk is completed, a **provisional** item→domain assignment can be supplied manually below.  
Leave empty to skip domain scoring until review is done.


In [ ]:
# ============================================================
# 7. QPEI FRAMEWORK + PROVISIONAL ITEM→DOMAIN MAP
# ============================================================

QPEI_FRAMEWORK = {
    "D1": {"name": "Teacher competence and pedagogical practice", "weight": 0.20},
    "D2": {"name": "Curriculum implementation and assessment", "weight": 0.15},
    "D3": {"name": "Learning environment and infrastructure", "weight": 0.15},
    "D4": {"name": "Student learning outcomes and FLN", "weight": 0.20},
    "D5": {"name": "Governance, management, and community support", "weight": 0.15},
    "D6": {"name": "Equity and inclusiveness", "weight": 0.15},
}
QPEI_WEIGHTS = {k: v["weight"] for k, v in QPEI_FRAMEWORK.items()}
assert abs(sum(QPEI_WEIGHTS.values()) - 1.0) < 1e-9

# ------------------------------------------------------------------
# PROVISIONAL MAP — fill after substantive review of the crosswalk.
# Format: {"tq1": "D1", "pq3": "D5", ...}
# Leave empty ({}) to run aggregation only (item-level school means)
# without domain/QPEI scores.
# ------------------------------------------------------------------
PROVISIONAL_ITEM_DOMAIN = {
    # Example (delete when real map is ready):
    # "tq1": "D1",
    # "tq2": "D1",
}

framework_df = pd.DataFrame([
    {"domain": d, "domain_name": s["name"], "weight": s["weight"]}
    for d, s in QPEI_FRAMEWORK.items()
])
display(framework_df)
framework_df.to_csv(TABLE_DIR / "R4_Table_06_QPEI_Framework_Weights.csv", index=False)

print(f"Provisional item→domain assignments: {len(PROVISIONAL_ITEM_DOMAIN)}")


# 8. School-level aggregation

For each instrument:

1. Keep items that are RETAIN_CANDIDATE (or all items if no decision log).
2. Aggregate respondent responses to **school means** (0–5 scale).
3. Optionally normalise to 0–100.
4. If a provisional domain map exists, compute domain scores and weighted QPEI.


In [ ]:
# ============================================================
# 8. SCHOOL-LEVEL ITEM AGGREGATION
# ============================================================

def retained_items(label):
    """Items to carry forward for this instrument."""
    all_items = ITEMS.get(label, [])
    if decision_log is None or "decision" not in getattr(decision_log, "columns", []):
        return all_items  # provisional: keep all
    sub = decision_log[decision_log["instrument"] == label]
    # Prefer final_decision if present, else decision
    col = "final_decision" if "final_decision" in sub.columns else "decision"
    keep = sub[sub[col].astype(str).str.upper().str.contains("RETAIN")]["item"].tolist()
    if not keep:
        # If nothing marked RETAIN yet, keep all and flag
        print(f"  {label}: no RETAIN items in decision log → using all items provisionally")
        return all_items
    return [i for i in all_items if i in keep]


school_item_means = {}

for label, spec in INSTRUMENTS.items():
    sid = school_id_map.get(label)
    if sid is None:
        print(f"{label}: skip aggregation (no school id)")
        continue

    items = retained_items(label)
    if not items:
        print(f"{label}: no items to aggregate")
        continue

    df = analysis[spec["sheet"]][[sid] + items].copy()
    means = df.groupby(sid)[items].mean()
    means.index.name = "school_id"
    means = means.reset_index()
    school_item_means[label] = means

    means.to_csv(TABLE_DIR / f"R4_School_ItemMeans_{label}.csv", index=False)
    print(f"{label:12s}: {len(means)} schools × {len(items)} items aggregated")


In [ ]:
# ============================================================
# 8A. DOMAIN SCORES + WEIGHTED QPEI (if map provided)
# ============================================================

def to_100(series, lo=1.0, hi=5.0):
    """Linear map from [lo, hi] to [0, 100]."""
    return ((series - lo) / (hi - lo) * 100).clip(0, 100)


if not PROVISIONAL_ITEM_DOMAIN:
    print("No provisional item→domain map supplied.")
    print("School item-means are saved; domain/QPEI scoring skipped until crosswalk is completed.")
    qpei_scores = None
else:
    # Collect all school-level item means into one wide table where possible
    # Prefer Teacher+Parent+Student for scoring; Observation/Environment optional
    pieces = []
    for label in ["Teacher", "Parent", "Student", "Observation", "Environment"]:
        if label not in school_item_means:
            continue
        m = school_item_means[label].set_index("school_id")
        pieces.append(m)

    if not pieces:
        qpei_scores = None
        print("No school item means available.")
    else:
        wide = pieces[0]
        for p in pieces[1:]:
            wide = wide.join(p, how="outer")

        # Domain scores = mean of available mapped items (on 0–100 scale)
        domain_scores = pd.DataFrame(index=wide.index)
        for domain in QPEI_FRAMEWORK:
            items_d = [i for i, d in PROVISIONAL_ITEM_DOMAIN.items() if d == domain and i in wide.columns]
            if not items_d:
                domain_scores[domain] = np.nan
                continue
            domain_scores[domain] = wide[items_d].apply(to_100).mean(axis=1)

        # Weighted QPEI (renormalise weights over non-missing domains per school)
        def weighted_qpei(row):
            vals, wts = [], []
            for d, w in QPEI_WEIGHTS.items():
                if pd.notna(row.get(d)):
                    vals.append(row[d])
                    wts.append(w)
            if not wts:
                return np.nan
            wts = np.array(wts)
            wts = wts / wts.sum()
            return float(np.dot(vals, wts))

        domain_scores["QPEI"] = domain_scores.apply(weighted_qpei, axis=1)
        domain_scores = domain_scores.reset_index()
        qpei_scores = domain_scores

        display(qpei_scores.sort_values("QPEI", ascending=False))
        qpei_scores.to_csv(TABLE_DIR / "R4_Table_07_School_Domain_QPEI_Scores.csv", index=False)
        print(f"\nSchools scored: {qpei_scores['QPEI'].notna().sum()}")


# 9. Sensitivity analysis skeleton

Compare theoretical weights vs equal weights (and optional alternatives).  
Rank correlations show whether school ordering is stable — key evidence that findings are not an artefact of the weighting scheme.


In [ ]:
# ============================================================
# 9. SENSITIVITY: THEORETICAL vs EQUAL vs ALTERNATIVE WEIGHTS
# ============================================================

if qpei_scores is None or "QPEI" not in getattr(qpei_scores, "columns", []):
    print("Sensitivity skipped — domain/QPEI scores not yet available.")
    print("Complete the item→domain map and re-run section 8A.")
else:
    domains = list(QPEI_WEIGHTS.keys())
    base = qpei_scores.set_index("school_id")[domains].copy()

    def apply_weights(df, weights):
        out = []
        for idx, row in df.iterrows():
            vals, wts = [], []
            for d, w in weights.items():
                if d in row and pd.notna(row[d]):
                    vals.append(row[d])
                    wts.append(w)
            if not wts:
                out.append(np.nan)
            else:
                wts = np.array(wts)
                wts = wts / wts.sum()
                out.append(float(np.dot(vals, wts)))
        return pd.Series(out, index=df.index, name="score")

    scenarios = {
        "theoretical": QPEI_WEIGHTS,
        "equal": {d: 1/6 for d in domains},
        # Optional alternative: slightly more weight on outcomes + teachers
        "alt_outcomes_focus": {"D1": 0.22, "D2": 0.13, "D3": 0.13, "D4": 0.25, "D5": 0.14, "D6": 0.13},
    }

    scores = pd.DataFrame({name: apply_weights(base, w) for name, w in scenarios.items()})
    ranks = scores.rank(ascending=False, method="average")

    print("Score correlations:")
    display(scores.corr(method="spearman"))
    print("\nRank correlations:")
    display(ranks.corr(method="spearman"))

    scores.to_csv(TABLE_DIR / "R4_Table_08_Sensitivity_Scores.csv")
    ranks.to_csv(TABLE_DIR / "R4_Table_09_Sensitivity_Ranks.csv")

    # Scatter: theoretical vs equal
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(scores["theoretical"], scores["equal"], alpha=0.75)
    lims = [min(scores.min().min(), 0), max(scores.max().max(), 100)]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("QPEI (theoretical weights)")
    ax.set_ylabel("QPEI (equal weights)")
    ax.set_title("Sensitivity: theoretical vs equal weights")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "R4_Figure_Sensitivity_Theoretical_vs_Equal.png", dpi=140)
    plt.show()


# 10. Validity-evidence checklist (for the manuscript)

This cell writes a structured checklist you can paste into the paper’s validity section.  
Tick items as evidence is completed.


In [ ]:
# ============================================================
# 10. VALIDITY EVIDENCE CHECKLIST
# ============================================================

checklist = [
    {"source": "Content validity", "evidence": "Literature-based 6-domain framework + 35 indicators", "status": "DONE"},
    {"source": "Content validity", "evidence": "Explicit distinction: conceptual selection vs empirical validation", "status": "DONE"},
    {"source": "Response-process", "evidence": "Translation verification / glossary", "status": "REVIEW"},
    {"source": "Response-process", "evidence": "Coding rules (1–5, 88, 99) documented and applied", "status": "DONE"},
    {"source": "Response-process", "evidence": "Reverse-coding based on item meaning", "status": "REVIEW"},
    {"source": "Data quality", "evidence": "Missingness audit by item and school", "status": "DONE"},
    {"source": "Data quality", "evidence": "School-ID consistency and coverage matrix", "status": "DONE (this notebook)"},
    {"source": "Data quality", "evidence": "Demographics / sample description", "status": "DONE (this notebook)"},
    {"source": "Internal structure", "evidence": "Ordinal EFA (multi-solution) on respondent instruments", "status": "DONE (Round 3)"},
    {"source": "Internal structure", "evidence": "Selective CFA on reflective subsets", "status": "DONE (Round 3)"},
    {"source": "Internal structure", "evidence": "Item decision log with retain/review/drop rationale", "status": "DONE (Round 3)"},
    {"source": "Crosswalk", "evidence": "35-indicator × 6-domain table with empirical + substantive decision", "status": "PENDING analyst review"},
    {"source": "Construct / index", "evidence": "School-level aggregation + domain scores + weighted QPEI", "status": "SKELETON (await map)"},
    {"source": "Construct / index", "evidence": "Sensitivity to alternative weights", "status": "SKELETON (await map)"},
    {"source": "Triangulation", "evidence": "Cross-source concordance (teacher/parent/student/obs/env)", "status": "PENDING"},
    {"source": "Practical validity", "evidence": "QPEI differentiates schools; identifies quality gaps", "status": "PENDING scores"},
]

checklist_df = pd.DataFrame(checklist)
display(checklist_df)
checklist_df.to_csv(TABLE_DIR / "R4_Table_10_Validity_Evidence_Checklist.csv", index=False)

(LOG_DIR / "R4_Validity_Checklist.md").write_text(
    "# QPEI Validity Evidence Checklist\n\n"
    + checklist_df.to_markdown(index=False)
    + "\n",
    encoding="utf-8",
)


In [ ]:
# ============================================================
# 11. ROUND 4 RESULTS JSON + MANIFEST
# ============================================================

def json_safe(obj):
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, (float, np.floating)):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.DataFrame):
        return {
            "type": "DataFrame",
            "rows": int(obj.shape[0]),
            "columns": [str(c) for c in obj.columns],
            "records": json_safe(obj.replace({np.nan: None}).to_dict(orient="records")),
        }
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    return str(obj)


RESULTS = {
    "metadata": {
        "analysis": "QPEI Round 4 — Demographics + Construction Path",
        "data": str(DATA_PATH),
        "generated": datetime.now().isoformat(timespec="seconds"),
    },
    "school_id_map": school_id_map,
    "coverage_n_schools": int(len(coverage)) if "coverage" in dir() else None,
    "provisional_item_domain_n": len(PROVISIONAL_ITEM_DOMAIN),
    "qpei_scored": qpei_scores is not None and "QPEI" in getattr(qpei_scores, "columns", []),
    "status": {
        "demographics_explored": True,
        "school_coverage_mapped": True,
        "aggregation_skeleton": True,
        "domain_map": "PROVISIONAL_EMPTY" if not PROVISIONAL_ITEM_DOMAIN else "SUPPLIED",
        "qpei_status": "SCORED" if (qpei_scores is not None and "QPEI" in getattr(qpei_scores, "columns", [])) else "AWAITING_CROSSWALK",
        "sensitivity_status": "RUN" if (qpei_scores is not None and "QPEI" in getattr(qpei_scores, "columns", [])) else "AWAITING_SCORES",
    },
    "output_files": [],
}

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        RESULTS["output_files"].append({
            "file": str(p.relative_to(OUTPUT_DIR)),
            "size_kb": round(p.stat().st_size / 1024, 2),
        })

results_path = OUTPUT_DIR / "results_round4.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(RESULTS), f, ensure_ascii=False, indent=2)

manifest = pd.DataFrame(RESULTS["output_files"])
if len(manifest):
    display(manifest)
manifest.to_csv(OUTPUT_DIR / "R4_OUTPUT_MANIFEST.csv", index=False)

print("=" * 100)
print("ROUND 4 COMPLETE")
print("=" * 100)
print("Output :", OUTPUT_DIR)
print("JSON   :", results_path)
print("Status :", RESULTS["status"])
print("\nNext:")
print("  1. Review demographic tables/figures for the sample-description section")
print("  2. Complete substantive crosswalk (item → domain)")
print("  3. Paste completed map into PROVISIONAL_ITEM_DOMAIN and re-run scoring + sensitivity")
print("  4. Add triangulation table across sources")


# Manuscript framing (reminder)

Do **not** centre the paper on “we found six factors.”

Centre it on:

> Development and empirical validation of a multidimensional **formative** Quality of Primary Education Index for Bangladesh.

Statistical contribution:

- Theory-driven six-domain framework  
- Systematic empirical screening of 35 indicators  
- Internal-structure analysis of respondent instruments (ordinal EFA/CFA where appropriate)  
- Retain/revise/drop decisions using statistical **and** substantive criteria  
- School-level aggregation, predetermined weights, sensitivity analysis  
- Triangulated evidence across teacher, parent, student, observation, and environment sources  

That is a defensible index paper — not a scale-validation paper that happens to produce an index.
